# 中医药明命名体识别任务

In [41]:
from torch.utils.data import Dataset as TorchDataset
import seqeval
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForSeq2Seq, TrainingArguments, Trainer)
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, TaskType


## 数据处理

In [2]:
import os
os.getcwd()

'C:\\Users\\hhm18\\Desktop\\course'

In [3]:
categories = set()

class MyDataProcFunc(TorchDataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
        self.dataset = self.all_to_bio(self.data)
    
    def load_data(self, data_file):
        Data = {}
        with open (data_file, "rt", encoding="utf-8") as f:
            # 文本使用空行进行分割句子
            for idx, line in enumerate(f.read().split("\n\n")):
                if not line:
                    break
                sentence, labels = "", []
                for i, item in enumerate(line.split("\n")):
                    char, tag = item.split(" ")
                    sentence += char
                    if tag.startswith("B"):
                        labels.append([i, i, char, tag[2:]])   # Remove the B- or I-
                        categories.add(tag[2:])
                    elif tag.startswith("I"):
                        labels[-1][1] = i
                        labels[-1][2] += char
                Data[idx] = {
                    "sentence" : sentence,
                    "labels" : labels
                }
        return Data
    
    def span_to_bio(self, sentence, spans):
        tokens = list(sentence)
        labels = ["O"] * len(tokens)
        for start, end, text, token_type in spans:
            labels[start] = f"B-{token_type}"
            for i in range(start + 1, end + 1):
                labels[i] = f"I-{token_type}"
        return {"tokens": tokens, "labels": labels}
    
    def all_to_bio(self, data):
        dataset_list = []
        for idx, item in data.items():
            sentence = item["sentence"]
            spans = item["labels"]
            bio_item = self.span_to_bio(sentence, spans)
            dataset_list.append(bio_item)
        return dataset_list
    
    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

In [ ]:
train_path = "./state3/dataset/ner_data/medical.train"
test_path = "./state3/dataset/ner_data/medical.test"

train_data = MyDataProcFunc(train_path)
for i in  range(3):
    print(train_data[i])

print("="*500)
test_data = MyDataProcFunc(test_path)
for i in  range(3):
    print(test_data[i])
dataset = Dataset.from_list(train_data)
dataset

{'tokens': ['现', '头', '昏', '口', '苦'], 'labels': ['O', 'O', 'O', 'B-临床表现', 'I-临床表现']}
{'tokens': ['目', '的', '观', '察', '复', '方', '丁', '香', '开', '胃', '贴', '外', '敷', '神', '阙', '穴', '治', '疗', '慢', '性', '心', '功', '能', '不', '全', '伴', '功', '能', '性', '消', '化', '不', '良', '的', '临', '床', '疗', '效'], 'labels': ['O', 'O', 'O', 'O', 'B-中医治疗', 'I-中医治疗', 'I-中医治疗', 'I-中医治疗', 'I-中医治疗', 'I-中医治疗', 'I-中医治疗', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'O', 'O', 'O', 'O', 'O']}
{'tokens': ['舒', '肝', '和', '胃', '消', '痞', '汤', '；', '功', '能', '性', '消', '化', '不', '良'], 'labels': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断', 'I-西医诊断']}
{'tokens': ['药', '进', '１', '０', '帖', '，', '黄', '疸', '稍', '退', '，', '饮', '食', '稍', '增', '，', '精', '神', '稍', '振'], 'labels': ['O', 'O', 'O', 'O', 'O', 'O', 'B-中医诊断', 'I-中医诊断', 'O', 'O', 'O', 'O', 'O', '

In [50]:
path = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(path)
print("tokenizer.eos_token is: ", tokenizer.eos_token,tokenizer.pad_token)

tokenizer.eos_token is:  <|im_end|> <|endoftext|>


In [11]:
# 获取标签集合（用于映射到id）
unique_labels = sorted({l for d in train_data for l in d["labels"]})
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
print(f"label2id: {label2id} \nid2label: {id2label}")

label2id: {'B-中医治则': 0, 'B-中医治疗': 1, 'B-中医证候': 2, 'B-中医诊断': 3, 'B-中药': 4, 'B-临床表现': 5, 'B-其他治疗': 6, 'B-方剂': 7, 'B-西医治疗': 8, 'B-西医诊断': 9, 'I-中医治则': 10, 'I-中医治疗': 11, 'I-中医证候': 12, 'I-中医诊断': 13, 'I-中药': 14, 'I-临床表现': 15, 'I-其他治疗': 16, 'I-方剂': 17, 'I-西医治疗': 18, 'I-西医诊断': 19, 'O': 20} 
id2label: {0: 'B-中医治则', 1: 'B-中医治疗', 2: 'B-中医证候', 3: 'B-中医诊断', 4: 'B-中药', 5: 'B-临床表现', 6: 'B-其他治疗', 7: 'B-方剂', 8: 'B-西医治疗', 9: 'B-西医诊断', 10: 'I-中医治则', 11: 'I-中医治疗', 12: 'I-中医证候', 13: 'I-中医诊断', 14: 'I-中药', 15: 'I-临床表现', 16: 'I-其他治疗', 17: 'I-方剂', 18: 'I-西医治疗', 19: 'I-西医诊断', 20: 'O'}


In [23]:
def proc_func(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        padding=True,
        is_split_into_words=True,
        max_length=512
    )
    
    word_ids = tokenized.word_ids()
    labels = []
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        else:
            labels.append(label2id[example["labels"][word_idx]])
    tokenized["labels"] = labels
    return tokenized

tokenized_dataset = dataset.map(proc_func, remove_columns=dataset.column_names)

Map:   0%|          | 0/5259 [00:00<?, ? examples/s]

In [24]:
tokenized_dataset

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 5259
})

In [25]:
tokenized_dataset["attention_mask"], tokenized_dataset["labels"], tokenized_dataset["input_ids"]

(Column([[1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]),
 Column([[20, 20, 20, 5, 15], [20, 20, 20, 20, 1, 11, 11, 11, 11, 11, 11, 20, 20, 20, 20, 20, 20, 20, 20, 20, 9, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 20, 20, 20, 20, 20], [20, 20, 20, 20, 20, 20, 20, 20, 9, 19, 19, 19, 19, 19, 19], [20, 20, 20, 20, 20, 5, 15, 20, 20, 20, 20, 20, 9, 19, 19, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20

In [53]:
tmp = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_data = tmp["train"]
valid_data = tmp["test"]
train_data, valid_data

(Dataset({
     features: ['labels', 'input_ids', 'attention_mask'],
     num_rows: 4733
 }),
 Dataset({
     features: ['labels', 'input_ids', 'attention_mask'],
     num_rows: 526
 }))

## 评估指标

In [ ]:
import seqeval
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

y_true = [['O', 'O', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'B-LOC', 'O'], ['B-PER', 'I-PER', 'O']]
y_pred = [['O', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'I-LOC', 'B-LOC', 'O'], ['B-PER', 'I-PER', 'O']]

result = classification_report(y_true, y_pred,  mode='strict', scheme=IOB2, output_dict=True)
print(result)
print(classification_report(y_true, y_pred,  mode='strict', scheme=IOB2))

{'LOC': {'precision': 0.5, 'recall': 0.5, 'f1-score': 0.5, 'support': 2}, 'PER': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 1}, 'micro avg': {'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'f1-score': 0.6666666666666666, 'support': 3}, 'macro avg': {'precision': 0.75, 'recall': 0.75, 'f1-score': 0.75, 'support': 3}, 'weighted avg': {'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'f1-score': 0.6666666666666666, 'support': 3}}
              precision    recall  f1-score   support

         LOC       0.50      0.50      0.50         2
         PER       1.00      1.00      1.00         1

   micro avg       0.67      0.67      0.67         3
   macro avg       0.75      0.75      0.75         3
weighted avg       0.67      0.67      0.67         3



In [38]:
def compute_ner_metrics(valid_preds):
    predictions, labels = valid_preds

    y_true = [[id2label[l] for l in label_seq if l != -100] for label_seq in labels]
    y_pred = [[id2label[p] for (p, l) in zip(pred_seq, label_seq) if l != -100] 
              for pred_seq, label_seq in zip(predictions, labels)]
    
    res = classification_report(y_true, y_pred, mode='strict', scheme=IOB2, output_dict=True)
    print(classification_report(y_true, y_pred,  mode='strict', scheme=IOB2))
    return {
    "precision/micro": res["micro avg"]["precision"],
    "recall/micro": res["micro avg"]["recall"],
    "f1/micro": res["micro avg"]["f1-score"],
    "precision/macro": res["macro avg"]["precision"],
    "recall/macro": res["macro avg"]["recall"],
    "f1/macro": res["macro avg"]["f1-score"],
    "precision/weighted": res["weighted avg"]["precision"],
    "recall/weighted": res["weighted avg"]["recall"],
    "f1/weighted": res["weighted avg"]["f1-score"],
    }

## 训练参数（PEFT/Training args）

In [44]:
model_path = "Qwen/Qwen3-0.6B"

model = AutoModelForTokenClassification.from_pretrained(
    model_path,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

train_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    target_modules=["q_proj",], 
)

model = get_peft_model(model, train_config)
model.print_trainable_parameters()

Some weights of Qwen3ForTokenClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.bias', 'score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 709,653 || all params: 596,781,098 || trainable%: 0.1189


In [45]:
model

PeftModelForTokenClassification(
  (base_model): LoraModel(
    (model): Qwen3ForTokenClassification(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_f

In [47]:
# config:
lr = 1e-5
num_epoch=5
per_device_train_batch_size=4
gradient_accumulation_steps=4
logging_steps=10
num_train_epochs=6
save_steps=200               
eval_steps=100

model.enable_input_require_grads()
model.gradient_checkpointing_enable()

In [ ]:
args = TrainingArguments(
        output_dir="./output/ner_qwen2.5_7B_lora_6",
        learning_rate=lr,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        logging_steps=logging_steps,
        num_train_epochs=num_train_epochs,
        save_steps=save_steps,                 
        save_on_each_node=True,
        gradient_checkpointing=True,
        logging_dir="../tf-logs/huanhuan6/rus",       
        report_to="tensorboard",
        eval_strategy="steps",
        eval_steps=eval_steps,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        bf16=True,
        max_grad_norm=1.0
    )

trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=valid_data,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

In [56]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 